# KX2 Barcode Reader

The KX2's onboard barcode reader is a serial 1D laser scanner wired to the controller PC. It's fully independent of the CAN motor stack, so it works even with the arm e-stopped — exposed as its own {class}`~pylabrobot.paa.kx2.KX2BarcodeReader` `Device`.

The model that ships with KX2 systems we've inspected is a **Denso MDI-4050** (firmware reports as `BD01J09`). Factory defaults: **9600 8N1**, no flow control. Find your port with `python -m serial.tools.list_ports -v`.

### macOS: USB-to-serial driver

The reader cable is typically a Prolific PL2303. macOS bundles a driver for older variants but not the newer GC/HXN chip (USB ID `067b:23a3`). If the device doesn't show up at `/dev/tty.PL2303G-USBtoUART<n>` after plugging it in:

```bash
brew install --cask prolific-pl2303
open -a PL2303Serial      # registers the system extension
```

Then **System Settings → Privacy & Security → Allow** on the "System software from PL2303Serial was blocked" prompt, restart if asked, and replug the cable. Verify with `systemextensionsctl list`: `com.prolific.cdc.PLCdcFSDriver` should show `[activated enabled]`.

### Symbologies

The MDI-4050 is a 1D laser, so **PDF417 / DataMatrix / QR Code aren't supported** — you'd need the 2D imager variants (e.g. MDI-4150) for those.

Typical out-of-the-box 1D enables: UPC-A, UPC-E, EAN-13, EAN-8, Code 39, Tri-Optic, Codabar, Industrial 2/5, Interleaved 2/5, IATA, Code 128, Code 93, GS1 DataBar, GS1 DataBar Limited. Add-on (supplemental) codes, postal codes, MSI/Plessey, Telepen, UK/Plessey, and Code 11 are off by default.

You can read your unit's full configuration over the wire — model, firmware, every per-symbology enable flag, prefixes/suffixes, and length limits — via `dump_config()` (which sends `Z4`):

```python
text = await bcr.driver.dump_config()
print(text[:200])  # header has MODEL / ROM Ver. / I/F
```

To **change** which symbologies are enabled, three options in order of practicality:

1. **Denso's MDI Setup utility** (free Windows download from Denso ADC) — checkbox UI, talks to the reader over the same serial port, writes to NVRAM.
2. **Configuration barcodes** printed in the MDI-4050 setting manual — scan a Start → toggle → End sequence; no host required.
3. Direct register writes via the ESC protocol — possible but the per-symbology byte layout is documented only in Denso's protocol manual.

{class}`~pylabrobot.resources.barcode.Barcode` returned from `scan()` has `symbology="ANY 1D"` by default — the reader doesn't emit a symbology ID unless prefix/suffix bytes are configured to include one.

## Setup

Pass the serial port path. `setup()` opens the port, does the `Z1` software-version handshake, and puts the reader into single-trigger mode.

In [ ]:
from pylabrobot.paa.kx2 import KX2BarcodeReader

bcr = KX2BarcodeReader(port="/dev/tty.PL2303G-USBtoUART11240")
await bcr.setup()


`scan(read_time=...)` fires the trigger and waits up to `read_time + 1s` for the reader to decode something. Omit `read_time` to use whatever read window is currently programmed on the reader. Returns a {class}`~pylabrobot.resources.barcode.Barcode`.

In [ ]:
barcode = await bcr.barcode_scanning.scan(read_time=8)
print(barcode.data)

Change the trigger mode at runtime:

In [ ]:
# "single" (default), "multiple" (two reads per trigger), or "continuous".
# await bcr.set_read_mode("continuous")

For raw vendor commands or the auto-trigger mode (reader fires on its own), go through `bcr.driver`:

In [ ]:
print(await bcr.driver.get_software_version())

# Full NVRAM dump — model, firmware, all symbology enables, prefixes/suffixes, length limits.
config = await bcr.driver.dump_config()
print(config[:200])

# await bcr.driver.set_auto_trigger(True)   # reader scans on its own
# await bcr.driver.set_auto_trigger(False)

## Teardown

`stop()` sends `Y` (trigger off) and `Y2` (reset to a 2s read window), then closes the serial port — leaving the reader in a known state for the next session.

In [ ]:
await bcr.stop()
